# Linear Regression — From Zero to Mastery

*OLS and MLE derivations, Gauss-Markov assumptions, robust/GLS corrections, Ridge/Lasso/Elastic Net regularization, and Bayesian linear regression — each concept built from scratch in NumPy, validated against sklearn/statsmodels, and illustrated with CAPM and Fama-French examples.*

## Table of Contents

- [0. Introduction](#0-introduction)
- [1. Univariate Linear Regression](#1-univariate-linear-regression)
  - [1.1 Problem Setup](#11-problem-setup)
  - [1.2 OLS — Ordinary Least Squares](#12-ols-ordinary-least-squares)
  - [1.3 MLE — Maximum Likelihood Estimation](#13-mle-maximum-likelihood-estimation)
  - [1.4 Residuals and Goodness of Fit](#14-residuals-and-goodness-of-fit)
- [2. Multivariate Regression](#2-multivariate-regression)
  - [2.1 Matrix Formulation and OLS](#21-matrix-formulation-and-ols)
  - [2.2 The Hat Matrix and Leverage](#22-the-hat-matrix-and-leverage)
  - [2.3 Adjusted R² and Model Selection](#23-adjusted-r2-and-model-selection)
  - [2.4 Multicollinearity and VIF](#24-multicollinearity-and-vif)
  - [2.5 Omitted Variable Bias](#25-omitted-variable-bias-ovb)
- [3. The Gauss-Markov Framework](#3-the-gauss-markov-framework)
  - [3.1 The Six Assumptions](#31-the-six-assumptions)
  - [3.2 Assumption Violations — Visual Diagnostics](#32-assumption-violations-visual-diagnostics)
  - [3.3 The Gauss-Markov Theorem](#33-the-gauss-markov-theorem)
  - [3.4 Inference — Standard Errors, t-tests, F-test](#34-inference-standard-errors-t-tests-f-test)
- [4. Assumption Violations and Fixes](#4-assumption-violations-and-fixes)
  - [4.0 GLS — The General Framework (WLS as special case)](#40-gls-the-general-framework)
  - [4.1 Heteroscedasticity](#41-heteroscedasticity)
  - [4.2 Autocorrelation and HAC Standard Errors](#42-autocorrelation-and-hac-standard-errors)
  - [4.3 Non-normality of Errors](#43-non-normality-of-errors)
- [5. Regularized Regression](#5-regularized-regression)
  - [5.1 Bias-Variance Tradeoff](#51-motivation-bias-variance-tradeoff)
  - [5.2 Ridge Regression (L2)](#52-ridge-regression-l2-penalty)
  - [5.3 Lasso Regression (L1)](#53-lasso-regression-l1-penalty)
  - [5.4 Elastic Net](#54-elastic-net)
- [6. Bayesian Linear Regression](#6-bayesian-linear-regression)
  - [6.1 The Bayesian Framework](#61-the-bayesian-framework)
  - [6.2 MAP Estimation = Regularization](#62-map-estimation-regularization)
  - [6.3 Full Bayesian Regression](#63-full-bayesian-regression)

In [1]:
from functions import run_cell as _run_cell; globals().update(_run_cell(2, globals()))


# 0. Introduction

## 0.1 What is Linear Regression?

Linear regression models the relationship between a **response variable** $y$ and one or more **predictors** $X$:

$$y = X\beta + \varepsilon$$

where $\beta$ are coefficients we estimate and $\varepsilon$ captures everything the model misses.

**Two distinct goals** — often confused, but they require different validation:

| Goal | Question | What matters |
|------|----------|-------------|
| **Prediction** | What will $y$ be for new $X$? | Out-of-sample error |
| **Inference** | What is the causal effect of $X$ on $y$? | Unbiased $\hat{\beta}$, valid standard errors |

**This guide covers both**, starting from first principles and building up to regularized and Bayesian variants.

## 0.2 Running Example: CAPM

The **Capital Asset Pricing Model** says a stock's excess return is linear in the market's excess return:

$$r_i - r_f = \alpha + \beta\,(r_m - r_f) + \varepsilon_i$$

- $\alpha$ (alpha): return above what the market explains — the manager's "edge"
- $\beta$ (beta): sensitivity to market movements (1 = moves with market, >1 = amplifies moves)
- $\varepsilon_i$: idiosyncratic (stock-specific) risk

This is a perfect example because the coefficients have clear economic interpretation.
In Section 2 we extend to the **Fama-French 3-factor model** (multivariate).

In [2]:
globals().update(run_cell(6, globals()))


NameError: name 'run_cell' is not defined

# 1. Univariate Linear Regression

## 1.1 Problem Setup

We have $n$ observations $(x_i, y_i)$ and want to find the **best linear fit**:

$$\hat{y}_i = \hat{\beta}_0 + \hat{\beta}_1\, x_i$$

The question is: what does "best" mean? Two principled answers:

1. **OLS** — geometric: minimize total squared distance between data and line (no distributional assumption)
2. **MLE** — probabilistic: find parameters most likely to have generated the data, *given a chosen distribution for $\varepsilon$*

MLE is a general principle — the distribution you choose determines the estimator you get:

| Error distribution | MLE loss | Resulting estimator |
|---|---|---|
| $\varepsilon \sim \mathcal{N}(0,\sigma^2)$ | $\sum(y_i - \hat{y}_i)^2$ = RSS | **Same $\hat{\beta}$ as OLS** |
| $\varepsilon \sim \text{Laplace}(0,b)$ | $\sum \lvert y_i - \hat{y}_i \rvert$ = LAD | Least Absolute Deviations (robust to outliers) |
| $\varepsilon \sim t_\nu$ | Huber-like loss | Robust regression |

Choosing Gaussian is not the only option — it just gives the cleanest connection to OLS.
We derive both principles below, then show why they agree under $\mathcal{N}$.
They give the **same $\hat{\beta}$** under Gaussian errors but illuminate different aspects.

## 1.2 OLS — Ordinary Least Squares

### 1.2.1 Why Squared Residuals, and Derivation

**The geometric question OLS asks**: among all possible lines $(\beta_0, \beta_1)$,
which one minimises the total squared vertical distance from the data points to the line?

Why squared (not absolute value)? Three reasons:
1. **Differentiable everywhere** — unlike $|e_i|$, which has a kink at 0, $e_i^2$ is smooth → we can set derivatives to zero and solve analytically
2. **Penalises large errors more** — a single huge error is much worse than many small ones; squared loss encodes this
3. **Unique closed-form solution** — the quadratic objective is strictly convex in $\beta$, guaranteeing a unique global minimum

**No statistical assumption is needed yet.** OLS is a purely geometric minimisation principle.

---

Minimise the **Residual Sum of Squares (RSS)**:

$$\text{RSS}(\beta_0, \beta_1) = \sum_{i=1}^{n}\left(y_i - \underbrace{\beta_0 + \beta_1 x_i}_{\hat{y}_i}\right)^2$$

Each term $(y_i - \hat{y}_i)^2$ is the squared vertical distance from data point $i$ to the fitted line.
The chart below shows two candidate lines — OLS finds the one where this total is smallest.

Set partial derivatives to zero:

$$\frac{\partial\,\text{RSS}}{\partial\,\beta_0} = -2\sum_{i=1}^{n}(y_i - \beta_0 - \beta_1 x_i) = 0
\;\implies\; \bar{y} = \hat{\beta}_0 + \hat{\beta}_1\,\bar{x} \tag{1}$$

$$\frac{\partial\,\text{RSS}}{\partial\,\beta_1} = -2\sum_{i=1}^{n} x_i(y_i - \beta_0 - \beta_1 x_i) = 0 \tag{2}$$

Substitute (1) into (2), centre variables ($\tilde{x}_i = x_i - \bar{x}$, $\tilde{y}_i = y_i - \bar{y}$):

$$\hat{\beta}_1 = \frac{\sum \tilde{x}_i\tilde{y}_i}{\sum \tilde{x}_i^2}$$

**Closed-form OLS estimators — the normal equations have an exact solution:**

$$\boxed{\hat{\beta}_1 = \frac{\widehat{\text{Cov}}(x,\,y)}{\widehat{\text{Var}}(x)}, \qquad \hat{\beta}_0 = \bar{y} - \hat{\beta}_1\,\bar{x}}$$

This is a closed-form solution — no iterative algorithm needed.
We extend this to matrix form in Section 2: $\hat{\beta} = (X'X)^{-1}X'y$.

In [ ]:
globals().update(run_cell(12, globals()))


### 1.2.2 Implementation and Validation

In [ ]:
globals().update(run_cell(14, globals()))


## 1.3 MLE — Maximum Likelihood Estimation

### 1.3.1 What MLE Is and How It Applies to Linear Regression

**MLE is a general statistical principle**, not specific to linear regression.
The idea: given observed data, find the parameters that make those data *most probable* under your assumed model.

**Three steps to apply MLE to linear regression:**

**Step 1 — Choose a distribution for the errors.**
Assume $\varepsilon_i \overset{iid}{\sim} \mathcal{N}(0,\,\sigma^2)$.
This is a modelling choice. We pick Gaussian because: it's analytically tractable, justified by the CLT in many settings, and — as we'll show — it leads back to OLS.
It says: given $x_i$ and $\beta$, the observation $y_i$ is drawn from:
$$y_i \mid x_i \;\sim\; \mathcal{N}(\beta_0 + \beta_1 x_i,\;\sigma^2)$$

**Step 2 — Write down the likelihood.**
The *likelihood* is the joint probability of observing all $n$ data points, viewed as a function of $(\beta, \sigma^2)$:
$$L(\beta, \sigma^2) = \prod_{i=1}^{n} p(y_i \mid x_i, \beta, \sigma^2)
= \prod_{i=1}^{n} \frac{1}{\sqrt{2\pi\sigma^2}}
\exp\!\left(-\frac{(y_i - \beta_0 - \beta_1 x_i)^2}{2\sigma^2}\right)$$

**Step 3 — Maximise the likelihood (or equivalently, the log-likelihood).**
The log is monotone, so $\arg\max L = \arg\max \log L$. Taking the log turns products into sums:
$$\ell(\beta, \sigma^2) = -\frac{n}{2}\log(2\pi\sigma^2)
- \underbrace{\frac{1}{2\sigma^2}\sum_{i=1}^{n}(y_i - \beta_0 - \beta_1 x_i)^2}_{\text{scaled RSS}}$$

For fixed $\sigma^2 > 0$, maximising $\ell$ w.r.t. $\beta$ is identical to **minimising RSS**.
This is the OLS criterion. So:

$$\boxed{\hat{\beta}^{\text{MLE}} = \hat{\beta}^{\text{OLS}} \quad \text{— the two principles yield the same estimator under Gaussian errors}}$$

**How MLE is solved — closed form vs gradient descent:**

In general, MLE has no closed form and requires iterative optimisation (gradient descent, Newton-Raphson, EM algorithm).
Here, the log-likelihood is *quadratic* in $\beta$ (same shape as RSS), so we can set $\nabla_\beta \ell = 0$ and solve analytically — giving the same closed-form solution as OLS.

**The closed-form solution is always preferred when it exists**: exact (no numerical error), instantaneous (no convergence iterations), and numerically stable. Gradient descent on this problem would waste computation. We demonstrate numerical optimisation below for completeness and to show they match.

### 1.3.2 OLS vs MLE — What Differs

Both give the same $\hat{\beta}$, but they differ in their assumptions, what else they estimate, and what they justify:

| | OLS | MLE (with Gaussian errors) |
|---|---|---|
| Error assumption | **None** — any distribution with finite variance | $\varepsilon \sim \mathcal{N}(0, \sigma^2)$ |
| $\hat{\beta}$ estimate | $\hat{\beta}^{\text{OLS}}$ | $= \hat{\beta}^{\text{OLS}}$ (same!) |
| $\hat{\sigma}^2$ | $\text{RSS}/(n-2)$ — **unbiased** (divides by degrees of freedom) | $\text{RSS}/n$ — **biased** (divides by $n$; underestimates) |
| How $\hat{\beta}$ is found | Set $\partial\text{RSS}/\partial\beta = 0$, solve analytically | Set $\partial\ell/\partial\beta = 0$, solve analytically (same equation) |
| Justification for $\hat{\beta}$ | Gauss-Markov: BLUE (Section 3) | Efficiency + asymptotic normality under $\mathcal{N}$ |
| Inference | Requires Normality (A6) or large $n$ via CLT | Exact finite-sample under $\mathcal{N}$ |
| Likelihood-based tools | ✗ | ✓ AIC, BIC, LR tests |

**When to use which:**
- Default: OLS — no distributional assumption needed for unbiasedness
- When you want model comparison (AIC/BIC) or build on this for GLMs: MLE framing is more natural

In [ ]:
globals().update(run_cell(18, globals()))


## 1.4 Residuals and Goodness of Fit

### 1.4.1 `R²` — Coefficient of Determination

In [ ]:
globals().update(run_cell(21, globals()))


Decompose `total variation` in $y$ into `explained` *and* `unexplained` parts:

$$\underbrace{\sum_{i}(y_i - \bar{y})^2}_{\text{SST (Total)}}
= \underbrace{\sum_{i}(\hat{y}_i - \bar{y})^2}_{\text{SSR (Regression)}}
+ \underbrace{\sum_{i}(y_i - \hat{y}_i)^2}_{\text{RSS (Residual)}}$$

$$R^2 = 1 - \frac{\text{RSS}}{\text{SST}} = \frac{\text{SSR}}{\text{SST}} \in [0, 1]$$

$R^2$ is the **fraction of variance in $y$ explained by the model**.

**OLS residuals always satisfy** (by construction of the normal equations):
1. $\sum e_i = 0$
2. $\sum x_i e_i = 0$ (residuals orthogonal to predictors)
3. The fitted line passes through $(\bar{x}, \bar{y})$

**Cautions**: high $R^2$ does not imply causation, correct specification, or good out-of-sample predictions.

### 1.4.2 Residual Diagnostics

**Always plot residuals before trusting any inference.** The diagnostic plots below are the primary tool for checking whether the Gauss-Markov assumptions (Section 3) hold:

| Plot | Axes & what each dot represents | Ideal look & why |
|------|--------------------------------|-----------------|
| **Residuals vs Fitted** | x = fitted value $\hat{y}_i$,  y = residual $e_i = y_i - \hat{y}_i$; each dot is one observation | **Flat random band around 0.** A curve → the true relationship is non-linear (A1 violated). A funnel (spread widens left-to-right) → variance is not constant (A5 violated, heteroscedasticity). |
| **Q-Q Plot** | x = theoretical quantile of $\mathcal{N}(0,1)$,  y = the corresponding quantile of the *standardised* residuals — i.e., sort residuals from smallest to largest and plot against where they'd fall if they were normal; each dot is one observation ranked by size | **Dots on the 45° diagonal.** That means each residual quantile matches the normal quantile — errors are Gaussian. Dots curving *up* in the tails → errors have heavier tails than normal (fat tails, common in finance). A systematic S-curve → skewed errors. Matters for the validity of t/F tests in small samples (A6). |
| **Residuals vs Index** | x = observation index (= time if data is time-ordered),  y = $e_i$; each dot is one time-ordered residual | **Random scatter, no runs or waves.** A wave pattern (long streaks of positive then negative residuals) signals AR(1) autocorrelation $e_t \approx \rho e_{t-1}$ — standard in financial return series. Violates A5; OLS SEs are underestimated → spurious significance. |
| **$\sqrt{\lvert\text{Residuals}\rvert}$ vs Fitted** | x = $\hat{y}_i$, y = $\sqrt{\lvert e_i \rvert}$; same layout as panel 1 but Y shows the *magnitude* of residuals (sign removed so trend direction is clear) | **Flat horizontal smoother.** An upward slope means observations with larger fitted values also have larger residuals → variance grows with $\hat{y}$ (classic in finance: larger stocks or higher-beta positions tend to have noisier returns). Violates A5. |

Under correct specification, residuals should look like random noise with **no pattern**.

In [ ]:
globals().update(run_cell(25, globals()))


# 2. Multivariate Regression

## 2.1 Matrix Formulation and OLS

### 2.1.1 Setup

With $n$ observations and $k-1$ predictors (plus intercept), stack everything into matrices:

$$\underbrace{y}_{n\times 1} = \underbrace{X}_{n\times k}\;\underbrace{\beta}_{k\times 1} + \underbrace{\varepsilon}_{n\times 1},
\qquad X = \begin{bmatrix}1 & x_{11} & \cdots & x_{1,k-1}\\
\vdots & \vdots & & \vdots \\ 1 & x_{n1} & \cdots & x_{n,k-1}\end{bmatrix}$$

The first column of $X$ is all-ones (for the intercept $\beta_0$).

### 2.1.2 OLS in Matrix Form

Minimise $\|y - X\beta\|^2$:

$$\frac{\partial}{\partial\beta}(y - X\beta)'(y - X\beta) = -2X'(y - X\beta) = 0
\implies X'X\hat{\beta} = X'y \quad \text{(Normal equations)}$$

Assuming $X$ has full column rank (A3), $(X'X)$ is invertible:

$$\boxed{\hat{\beta} = (X'X)^{-1}X'y}$$

The univariate formula is a special case of this.

**Running example: Fama-French 3-Factor Model**

$$r_i - r_f = \alpha + \beta_{\text{MKT}}(r_m - r_f) + \beta_{\text{SMB}}\cdot\text{SMB} + \beta_{\text{HML}}\cdot\text{HML} + \varepsilon_i$$

- **SMB** (Small Minus Big): size premium — small-cap stocks outperform large-cap
- **HML** (High Minus Low): value premium — high book-to-market stocks outperform growth

In [ ]:
globals().update(run_cell(29, globals()))


## 2.2 The Hat Matrix and Leverage

### 2.2.1 The Projection Matrix

The OLS fitted values are a **linear projection** of $y$ onto the column space of $X$:

$$\hat{y} = X\hat{\beta} = X(X'X)^{-1}X'y \equiv Hy$$

where $H \equiv X(X'X)^{-1}X'$ is the **hat matrix** (it "puts a hat on $y$").

**Properties of $H$**:
- **Symmetric**: $H' = H$
- **Idempotent**: $H^2 = H$ (projecting twice = projecting once)
- $Hy = \hat{y}$ (gives fitted values)
- $(I-H)y = e$ (residuals are the orthogonal complement)
- $\text{tr}(H) = k$ (sum of diagonal = number of parameters)

### 2.2.2 Leverage

The diagonal elements $h_{ii}$ are called **leverage scores**:

$$h_{ii} = [H]_{ii} = x_i'(X'X)^{-1}x_i, \qquad \frac{1}{n} \leq h_{ii} \leq 1$$

High leverage ($h_{ii}$ close to 1) means observation $i$ has **outsized influence** on the fitted line.
An observation can have high leverage without being an outlier (it depends only on $X$, not on $y$).

### 2.2.3 Cook's Distance

Combines leverage **and** residual size into a single influence measure:

$$D_i = \frac{e_i^2}{k\hat{\sigma}^2}\cdot\frac{h_{ii}}{(1-h_{ii})^2}$$

A large $D_i$ (rule of thumb: $D_i > 4/n$) flags observation $i$ as influential — removing it would substantially change $\hat{\beta}$.

In [ ]:
globals().update(run_cell(32, globals()))


## 2.3 Adjusted R² and Model Selection

### 2.3.1 The Problem with R²

In multivariate regression, adding any predictor — even a random noise variable — **always increases R²**.
This makes R² a misleading criterion for comparing models with different numbers of predictors.

### 2.3.2 Adjusted R²

Penalise for the number of parameters $k$:

$$\bar{R}^2 = 1 - \frac{\text{RSS}/(n-k)}{\text{SST}/(n-1)} = 1 - (1 - R^2)\cdot\frac{n-1}{n-k}$$

$\bar{R}^2$ **can decrease** when adding a weak predictor. It's a better model selection criterion than $R^2$.

### 2.3.3 Information Criteria (AIC, BIC)

For likelihood-based comparison (especially comparing models with different functional forms):

$$\text{AIC} = 2k - 2\hat{\ell}, \qquad \text{BIC} = k\log n - 2\hat{\ell}$$

where $\hat{\ell}$ is the maximised log-likelihood. **Lower is better.**
BIC penalises complexity more heavily than AIC (relevant for large $n$).

In [ ]:
globals().update(run_cell(35, globals()))


## 2.4 Multicollinearity and VIF

### 2.4.1 What is Multicollinearity?

When predictors are **highly correlated**, the columns of $X$ are near-linearly-dependent.
$(X'X)^{-1}$ still exists (no perfect collinearity = A3 holds), but it has very large entries.

Consequence: $\text{Var}(\hat{\beta}) = \sigma^2(X'X)^{-1}$ becomes **inflated** — huge standard errors,
wide confidence intervals, and $t$-statistics near zero even when the true $\beta \neq 0$.

Intuitively: with two highly correlated predictors $x_1$ and $x_2$, the data cannot "decide" how to
split the explanatory power between them — any combination $a\beta_1 - b\beta_2$ fits the data almost
equally well, making individual estimates unstable.

### 2.4.2 Variance Inflation Factor (VIF)

For predictor $j$, regress $x_j$ on all other predictors. Let $R^2_j$ be the resulting R²:

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

$R^2_j$ close to 1 means $x_j$ is nearly explained by the others → high VIF → inflated variance.

**Rules of thumb**: VIF > 5 warrants investigation; VIF > 10 is severe.

**Fixes**: remove one of the correlated predictors, combine them (PCA), or use Ridge regression (Section 5).

In [ ]:
globals().update(run_cell(38, globals()))


## 2.5 Omitted Variable Bias (OVB)

### 2.5.1 The Problem

Suppose the true model is $y = X\beta + Z\gamma + \varepsilon$ where $Z$ is an **omitted variable**.
If we estimate the short regression $y = X\tilde{\beta} + \tilde{\varepsilon}$:

$$\mathbb{E}[\hat{\tilde{\beta}}] = \beta + \underbrace{(X'X)^{-1}X'Z\gamma}_{\text{OVB}}$$

The OVB term is the product of:
1. $(X'X)^{-1}X'Z$ — the coefficient from regressing $Z$ on $X$ ("how correlated is $Z$ with $X$?")
2. $\gamma$ — the true effect of $Z$ on $y$

**OVB = 0 when**:
- $Z$ is uncorrelated with $X$ → $(X'X)^{-1}X'Z = 0$, OR
- $Z$ has no effect on $y$ → $\gamma = 0$

Otherwise $\hat{\beta}$ is biased — this violates A4.

### 2.5.2 Direction of OVB (univariate case)

$$\text{OVB} = \underbrace{\text{Cov}(x, z)/\text{Var}(x)}_{\text{sign of }x\text{-}z\text{ corr}}\times \underbrace{\gamma}_{\text{sign of }z\text{'s effect}}$$

| $\text{Corr}(x,z)$ | $\gamma$ | OVB direction |
|---|---|---|
| + | + | Upward bias (overestimate $\beta$) |
| + | − | Downward bias |
| − | + | Downward bias |
| − | − | Upward bias |

### 2.5.3 Finance Example

Regressing a stock's return on the market factor (CAPM) while **omitting the size factor** (SMB):
if the stock loads positively on both market and SMB ($\beta_{\text{SMB}} > 0$), and if market and SMB
are positively correlated, then the OLS $\hat{\beta}_{\text{MKT}}$ is **upward biased**.

In [ ]:
globals().update(run_cell(41, globals()))


# 3. The Gauss-Markov Framework

## 3.1 The Six Assumptions

Now that we can write OLS in matrix form and interpret the hat matrix, we ask: **when is OLS a good estimator?**
"Good" means two things: (1) it gives the right answer on average (unbiasedness) and (2) it uses the data efficiently (minimum variance). The Gauss-Markov (GM) assumptions specify exactly the conditions needed.

Each assumption either makes OLS *work* (A1-A4 = unbiasedness) or makes OLS *optimal* (A5 = efficiency, A6 = exact inference).

### 3.1.1 The Six Assumptions

The power of OLS comes with conditions. Here are the Gauss-Markov (GM) assumptions — what each requires and what it gives us:

| # | Name | Statement | What it gives us |
|---|------|-----------|-----------------|
| **A1** | **Linearity** | $y = X\beta + \varepsilon$ (linear in $\beta$) | Model correctly specified |
| **A2** | **Random sampling** | $(x_i, y_i) \overset{iid}{\sim}$ population | Estimates generalise |
| **A3** | **No perfect multicollinearity** | $\text{rank}(X) = k$ (columns linearly independent) | $(X'X)^{-1}$ exists, OLS defined |
| **A4** | **Zero conditional mean (exogeneity)** | $\mathbb{E}[\varepsilon \mid X] = 0$ | OLS is **unbiased**: $\mathbb{E}[\hat{\beta}] = \beta$ |
| **A5** | **Spherical errors (homoscedasticity)** | $\text{Var}(\varepsilon \mid X) = \sigma^2 I_n$ | OLS is **efficient** (min-variance among linear unbiased) |
| **A6** | **Normality for error terms** | $\varepsilon \mid X \sim \mathcal{N}(0, \sigma^2 I_n)$ | **Exact** finite-sample inference ($t$, $F$ distributions) |

**A4 is the most critical.** Without it, $\hat{\beta}$ is biased — the estimates are wrong in expectation.
This is why endogeneity (omitted variable, simultaneity, measurement error) is so damaging.

**A5 has two sub-conditions:**
- **Homoscedasticity**: $\text{Var}(\varepsilon_i) = \sigma^2$ for all $i$ (same variance)
- **No autocorrelation**: $\text{Cov}(\varepsilon_i, \varepsilon_j) = 0$ for $i \neq j$

Violating A5 does **not** bias $\hat{\beta}$, but makes standard errors wrong — invalidating all $t$-tests and $F$-tests.

**A6 is only needed for finite-sample inference.** By the CLT, tests are asymptotically valid without it.

In [ ]:
globals().update(run_cell(45, globals()))


## 3.2 Assumption Violations — Visual Diagnostics

The residual plot (residuals vs fitted values) is the primary diagnostic tool.
**Under correct specification, residuals should look like random noise — no pattern whatsoever.**

Below we simulate four canonical violations and show what they look like in the residual plot.

In [ ]:
globals().update(run_cell(48, globals()))


## 3.3 The Gauss-Markov Theorem

> **Under assumptions A1–A5, OLS is BLUE: the Best Linear Unbiased Estimator.**

This is the theorem that *justifies* OLS. Break down the acronym:

- **Linear**: $\hat{\beta} = Cy$ for some matrix $C$ (a linear function of the data $y$)
- **Unbiased**: $\mathbb{E}[\hat{\beta}] = \beta$ (right on average)
- **Best**: minimum variance among all linear unbiased estimators

### Proof sketch

OLS: $\hat{\beta} = (X'X)^{-1}X'y$. Substituting $y = X\beta + \varepsilon$:

$$\hat{\beta} = (X'X)^{-1}X'(X\beta + \varepsilon) = \beta + \underbrace{(X'X)^{-1}X'\varepsilon}_{\text{estimation error}}$$

**Unbiasedness** (uses A4: $\mathbb{E}[\varepsilon\mid X]=0$):

$$\mathbb{E}[\hat{\beta}\mid X] = \beta + (X'X)^{-1}X'\underbrace{\mathbb{E}[\varepsilon\mid X]}_{=\,0} = \beta \checkmark$$

**Variance** (uses A5: $\text{Var}(\varepsilon\mid X)=\sigma^2 I$):

$$\text{Var}(\hat{\beta}\mid X)
= (X'X)^{-1}X'\,\sigma^2 I\,X(X'X)^{-1}
= \sigma^2(X'X)^{-1}$$

**Optimality**: for any other linear unbiased estimator $\tilde{\beta} = Cy$ (with $CX = I$), one can show
$\text{Var}(\tilde{\beta}) - \text{Var}(\hat{\beta}) \succeq 0$ (positive semi-definite) — OLS has no more variance than any competitor.

### What breaks when assumptions fail

| Violated | Consequence |
|---------|-------------|
| **A4** (endogeneity) | $\hat{\beta}$ **biased** — estimates are systematically wrong |
| **A5** (heterosk./autocorr.) | $\hat{\beta}$ unbiased but **not efficient**; standard errors wrong → invalid tests |
| **A3** (perfect multicollin.) | $(X'X)^{-1}$ undefined — OLS does not exist |
| **A6** (non-normality) | $t$/$F$ distributions only approximate; OK asymptotically via CLT |

## 3.4 Inference — Standard Errors, t-tests, F-test

### 3.4.1 Distribution of $\hat{\beta}$ and Standard Errors

Under A1–A6, the OLS estimator has an **exact normal distribution**:

$$\hat{\beta} \mid X \sim \mathcal{N}\!\left(\beta,\;\sigma^2(X'X)^{-1}\right)$$

Since $\sigma^2$ is unknown, replace it with $\hat{\sigma}^2 = \text{RSS}/(n-k)$ (where $k$ = number of coefficients):

$$\frac{\hat{\beta}_j - \beta_j}{\widehat{\text{SE}}(\hat{\beta}_j)} \sim t_{n-k}$$

For the univariate case:

$$\widehat{\text{SE}}(\hat{\beta}_1) = \frac{\hat{\sigma}}{\sqrt{\sum(x_i - \bar{x})^2}},
\qquad
\widehat{\text{SE}}(\hat{\beta}_0) = \hat{\sigma}\sqrt{\frac{\sum x_i^2}{n\sum(x_i-\bar{x})^2}}$$

**Factors that shrink SE (more precision)**:
- Larger $n$ (more data)
- Larger spread of $x$ values ($\sum(x_i-\bar{x})^2$ in denominator)
- Smaller error variance $\sigma^2$

### 3.4.2 t-tests, F-test, and Confidence Intervals

**t-test for a single coefficient**
$H_0: \beta_j = 0$ vs $H_1: \beta_j \neq 0$

$$t_j = \frac{\hat{\beta}_j}{\widehat{\text{SE}}(\hat{\beta}_j)} \;\overset{H_0}{\sim}\; t_{n-k}$$

**F-test for joint significance** (all slope coefficients are zero):
$H_0: \beta_1 = \cdots = \beta_{k-1} = 0$

$$F = \frac{\text{SSR}/(k-1)}{\text{RSS}/(n-k)} \;\overset{H_0}{\sim}\; F_{k-1,\;n-k}$$

Note: in univariate regression, $F = t^2$ (equivalent tests).

**95% Confidence Interval**:

$$\hat{\beta}_j \pm t_{n-k,\;0.025} \cdot \widehat{\text{SE}}(\hat{\beta}_j)$$

The 95% CI does **not** mean "95% probability that $\beta$ is in this interval" — $\beta$ is fixed (frequentist).
It means: if we repeated the experiment many times, 95% of the constructed intervals would contain $\beta$.

In [ ]:
globals().update(run_cell(56, globals()))


# 4. Assumption Violations and Fixes

## 4.0 GLS — The General Framework

Before diving into specific violations, let's establish the **general solution** for when the spherical-errors assumption (A5) fails.

### 4.0.1 The Problem

Suppose the true variance-covariance matrix of the errors is:
$$\text{Var}(\varepsilon \mid X) = \sigma^2\,\Omega$$

where $\Omega$ is an $n \times n$ positive definite matrix (known or estimable).

- **Homoscedastic, no autocorrelation** (A5 holds): $\Omega = I$
- **Heteroscedastic, no autocorrelation**: $\Omega = \text{diag}(\omega_1, \ldots, \omega_n)$ (diagonal, different entries)
- **Autocorrelation** (e.g., AR(1) errors): $\Omega$ has off-diagonal entries

Under $\Omega \neq I$, OLS is **unbiased but inefficient** — there exists a better linear unbiased estimator.

### 4.0.2 GLS — Generalised Least Squares

**Key idea**: pre-multiply the model by $\Omega^{-1/2}$ to transform it into a homoscedastic one:

$$\underbrace{\Omega^{-1/2}y}_{\tilde{y}} = \underbrace{\Omega^{-1/2}X}_{\tilde{X}}\,\beta + \underbrace{\Omega^{-1/2}\varepsilon}_{\tilde{\varepsilon}}$$

The transformed errors satisfy $\text{Var}(\tilde{\varepsilon}\mid X) = \sigma^2 I$ — spherical errors! Apply OLS to this transformed system:

$$\boxed{\hat{\beta}^{\text{GLS}} = (\tilde{X}'\tilde{X})^{-1}\tilde{X}'\tilde{y} = (X'\Omega^{-1}X)^{-1}X'\Omega^{-1}y}$$

**Gauss-Markov for the transformed model**: $\hat{\beta}^{\text{GLS}}$ is BLUE in the model with $\text{Var}(\varepsilon)=\sigma^2\Omega$. OLS is no longer efficient; GLS is.

### 4.0.3 WLS — Weighted Least Squares (Special Case of GLS)

When $\Omega = \text{diag}(1/w_1, \ldots, 1/w_n)$ (heteroscedastic, no autocorrelation):

$$\hat{\beta}^{\text{WLS}} = \arg\min_\beta \sum_{i=1}^n w_i(y_i - x_i'\beta)^2 = (X'WX)^{-1}X'Wy, \quad W = \text{diag}(w_i)$$

Observations with higher weight $w_i$ have lower variance $\sigma^2/w_i$ — we trust them more.

| Setting | $\Omega$ | Solution |
|---------|---------|---------|
| Heteroscedastic ($\sigma_i^2 = \sigma^2/w_i$) | $\text{diag}(1/w_i)$ | WLS with weights $w_i$ |
| AR(1) errors ($\varepsilon_t = \rho\varepsilon_{t-1}+\nu_t$) | Toeplitz matrix of $\rho^{|i-j|}$ | Cochrane-Orcutt / Prais-Winsten |
| General $\Omega$ | Any PD matrix | GLS: $(X'\Omega^{-1}X)^{-1}X'\Omega^{-1}y$ |

### 4.0.4 FGLS — Feasible GLS

In practice, $\Omega$ is **unknown**. FGLS estimates $\Omega$ in a first step, then applies GLS:
1. Run OLS, get residuals $e_i$
2. Estimate $\Omega$ from residuals (e.g., model $\hat{\sigma}_i^2 = f(x_i)$ for heteroscedasticity, or fit an AR model to $e_t$)
3. Apply GLS with $\hat{\Omega}$

**Caution**: FGLS is efficient only if $\hat{\Omega}$ is a good estimate. If $\Omega$ is misspecified, FGLS can be **worse** than OLS.
This is why robust standard errors (HC, HAC) are often preferred in practice: they don't require knowing $\Omega$, only give valid inference (not efficiency).

In [ ]:
globals().update(run_cell(59, globals()))


## 4.1 Heteroscedasticity

**Violation**: $\text{Var}(\varepsilon_i \mid x_i) = \sigma_i^2$ (not constant $\sigma^2$).

Very common in financial data — return volatility changes over time (e.g., higher variance in periods of market stress).

### 4.1.1 Consequences

$\hat{\beta}^{\text{OLS}}$ remains **unbiased** (A4 still holds), but:
- OLS is no longer efficient (not BLUE — Gauss-Markov fails)
- The standard error formula $\hat{\sigma}^2(X'X)^{-1}$ is **wrong** → $t$-stats and $F$-stats are invalid

### 4.1.2 Detection

- **Visual**: fan shape in residuals-vs-fitted plot (variance grows with $\hat{y}$)
- **Breusch-Pagan test**: regress $e_i^2$ on $X$, test if coefficients are jointly zero; under $H_0$: $nR^2_{\text{aux}} \sim \chi^2_{k-1}$
- **White test**: more general version of Breusch-Pagan

### 4.1.3 Fixes

**Fix 1 — Heteroscedasticity-Robust (HC) Standard Errors** (White 1980):

$$\widehat{\text{Var}}_{\text{HC}}(\hat{\beta}) = (X'X)^{-1}\left(\sum_{i=1}^{n} e_i^2 x_i x_i'\right)(X'X)^{-1}$$

Keep OLS $\hat{\beta}$ but replace SEs with robust ones. No efficiency gain, but valid inference.

**Fix 2 — Weighted Least Squares (WLS)**:

$$\hat{\beta}_{\text{WLS}} = \arg\min_\beta \sum_{i=1}^{n} w_i(y_i - x_i'\beta)^2, \quad w_i = 1/\sigma_i^2$$

When $\sigma_i^2$ is known or estimable, WLS is efficient (BLUE in the heteroscedastic world).

In [ ]:
globals().update(run_cell(62, globals()))


## 4.2 Autocorrelation and HAC Standard Errors

**Violation**: $\text{Cov}(\varepsilon_t, \varepsilon_{t-j}) \neq 0$ for $j \neq 0$.

Extremely common in financial **time series** — stock returns, macro variables, volatility.

### 4.2.1 Consequences

Same as heteroscedasticity: $\hat{\beta}$ remains **unbiased** (if A4 holds), but standard errors are wrong.
Typically **underestimated** with positive autocorrelation → spuriously significant $t$-tests.

### 4.2.2 Detection

**Durbin-Watson statistic**: tests for first-order autocorrelation.
$$DW = \frac{\sum_{t=2}^{n}(e_t - e_{t-1})^2}{\sum_{t=1}^n e_t^2} \approx 2(1 - \hat{\rho})$$

- $DW \approx 2$: no autocorrelation
- $DW < 2$: positive autocorrelation ($DW \approx 0$ means $\hat{\rho} \approx 1$)
- $DW > 2$: negative autocorrelation

Also: plot the residual ACF (autocorrelation function).

### 4.2.3 Fix — Newey-West HAC Standard Errors

**HAC** (Heteroscedasticity and Autocorrelation Consistent) standard errors, also called Newey-West:

$$\widehat{\text{Var}}_{\text{NW}}(\hat{\beta}) = (X'X)^{-1}\hat{\Omega}(X'X)^{-1}$$

$$\hat{\Omega} = \sum_{i=1}^n e_i^2 x_i x_i' + \sum_{j=1}^{L}\left(1 - \frac{j}{L+1}\right)\sum_{t=j+1}^{n}(e_t e_{t-j})(x_t x_{t-j}' + x_{t-j}x_t')$$

where $L$ is the **bandwidth** (number of lags to include; rule of thumb: $L \approx n^{1/3}$).

Newey-West SEs are the **standard tool in empirical finance** for time-series regressions.

In [ ]:
globals().update(run_cell(65, globals()))


## 4.3 Non-normality of Errors

**Violation**: $\varepsilon \mid X \not\sim \mathcal{N}(0, \sigma^2 I)$ (assumption A6 fails).

### 4.3.1 Consequences and When It Matters

Under A1–A5 (not A6):
- $\hat{\beta}$ is still BLUE (GM theorem doesn't need normality)
- By the **CLT**, as $n \to \infty$: $\hat{\beta} \overset{d}{\to} \mathcal{N}(\beta, \sigma^2(X'X)^{-1})$

So non-normality is only an issue for **small samples** (exact $t$/$F$ distributions don't hold).
For $n > 50$–100, asymptotic results are usually adequate.

**In finance**: return distributions typically have **heavy tails** (kurtosis > 3). The key implication:
standard inference may underestimate tail risk. For risk management, this matters more than for regression inference.

### 4.3.2 Detection

**Q-Q Plot**: points should lie on a 45° line. Deviations in the tails → non-normality.
- S-shaped deviation → heavy tails (leptokurtosis, e.g., $t$-distribution)
- Opposite S-shape → thin tails

**Jarque-Bera test**: tests skewness and kurtosis jointly.
$$JB = \frac{n}{6}\left(S^2 + \frac{(K-3)^2}{4}\right) \overset{H_0}{\sim} \chi^2_2$$

where $S$ = sample skewness, $K$ = sample kurtosis. Rejects $H_0$ of normality for large $JB$.

In [ ]:
globals().update(run_cell(68, globals()))


# 5. Regularized Regression

## 5.1 Motivation: Bias-Variance Tradeoff

### 5.1.1 When Does OLS Fail?

OLS has minimum variance among **unbiased** estimators. But being unbiased isn't always optimal.
If we allow a small bias, we can sometimes **dramatically reduce variance** — and get lower MSE overall.

$$\text{MSE}(\hat{\beta}) = \underbrace{\text{Bias}(\hat{\beta})^2}_{\text{systematic error}} + \underbrace{\text{Var}(\hat{\beta})}_{\text{random error}}$$

OLS is unbiased → Bias = 0, but Var can be large when:
- **$p \approx n$**: many predictors relative to observations (near-multicollinearity or exactly $p = n$, OLS overfits)
- **High multicollinearity**: $(X'X)^{-1}$ has large entries → inflated variance
- **Irrelevant predictors**: adding noise variables inflates variance without bias reduction

**Regularization trades a little bias for a lot of variance reduction**, giving lower total MSE.

### 5.1.2 General Form

Regularised regression adds a **penalty** $\Omega(\beta)$ to the loss:

$$\hat{\beta}_{\lambda} = \arg\min_\beta \underbrace{\|y - X\beta\|^2}_{\text{RSS}} + \lambda\,\Omega(\beta)$$

- $\lambda = 0$: recover OLS
- $\lambda \to \infty$: $\hat{\beta} \to 0$ (extreme shrinkage)
- Choosing $\lambda$: cross-validation

In [ ]:
globals().update(run_cell(72, globals()))


## 5.2 Ridge Regression (L2 Penalty)

### 5.2.1 Objective

Add an **L2 penalty** (sum of squared coefficients):

$$\hat{\beta}^{\text{Ridge}}_\lambda = \arg\min_\beta \|y - X\beta\|^2 + \lambda\|\beta\|_2^2$$

Note: typically **do not penalise the intercept** $\beta_0$; standardise predictors first.

### 5.2.2 Closed-Form Solution

$$\frac{\partial}{\partial\beta}\left[\|y - X\beta\|^2 + \lambda\|\beta\|^2\right]
= -2X'(y - X\beta) + 2\lambda\beta = 0$$

$$\boxed{\hat{\beta}^{\text{Ridge}} = (X'X + \lambda I)^{-1}X'y}$$

Compare to OLS: $\hat{\beta}^{\text{OLS}} = (X'X)^{-1}X'y$. Ridge adds $\lambda I$ to the diagonal of $X'X$ before inverting — this **regularises** the matrix (makes it invertible even when $X'X$ is nearly singular).

### 5.2.3 Properties

- **Always shrinks toward zero**: $\|\hat{\beta}^{\text{Ridge}}\| < \|\hat{\beta}^{\text{OLS}}\|$
- **Never zeros out**: all coefficients remain non-zero (L2 ball has no corners)
- **Biased**: $\mathbb{E}[\hat{\beta}^{\text{Ridge}}] \neq \beta$, but variance is reduced
- **Handles multicollinearity**: $(X'X + \lambda I)$ is always invertible for $\lambda > 0$

### 5.2.4 Geometric Intuition

Ridge constrained form: $\min_\beta \|y - X\beta\|^2$ subject to $\|\beta\|_2^2 \leq t$.

The **L2 constraint region** is a **sphere** — the OLS solution is projected onto it.
The solution is continuous and smooth — no exact zeros.

In [ ]:
globals().update(run_cell(75, globals()))


## 5.3 Lasso Regression (L1 Penalty)

### 5.3.1 Objective

Add an **L1 penalty** (sum of absolute coefficients):

$$\hat{\beta}^{\text{Lasso}}_\lambda = \arg\min_\beta \|y - X\beta\|^2 + \lambda\|\beta\|_1$$

### 5.3.2 No Closed Form

The L1 penalty is **not differentiable at zero** (subgradient needed), so there is no matrix-inverse solution.
Algorithms: **coordinate descent** (update one $\beta_j$ at a time, analytically tractable), LARS, proximal gradient.

For orthogonal $X$ (simplified case), the Lasso solution has a closed form via **soft-thresholding**:

$$\hat{\beta}^{\text{Lasso}}_j = \text{sign}(\hat{\beta}^{\text{OLS}}_j)\cdot\max\!\left(|\hat{\beta}^{\text{OLS}}_j| - \frac{\lambda}{2},\; 0\right)$$

Coefficients are **zeroed out exactly** when small enough — this is the key property.

### 5.3.3 Sparsity — Why Lasso Sets Coefficients Exactly to Zero

**Geometric intuition**: the L1 constraint region is a **diamond** (in 2D). Its corners lie on the axes.
The RSS ellipses (OLS contours) are most likely to first touch the diamond at a corner → corner = one coordinate is zero.

**Algorithmic intuition**: coordinate descent zeros out $\beta_j$ when $|X_j'r_{-j}| \leq \lambda/2$
(where $r_{-j}$ = residuals excluding $j$'s contribution) — a hard threshold.

### 5.3.4 Ridge vs Lasso

| | Ridge | Lasso |
|---|---|---|
| Penalty | $\lambda\|\beta\|_2^2$ | $\lambda\|\beta\|_1$ |
| Closed form | Yes: $(X'X+\lambda I)^{-1}X'y$ | No (coordinate descent) |
| Sparsity | No — shrinks all | Yes — can zero out |
| Best when | All predictors relevant (dense signal) | Many irrelevant predictors (sparse signal) |
| Correlated predictors | Splits weight among them | Picks one arbitrarily |

In [ ]:
globals().update(run_cell(78, globals()))


## 5.4 Elastic Net

### 5.4.1 Combining L1 and L2

$$\hat{\beta}^{\text{EN}} = \arg\min_\beta \|y - X\beta\|^2 + \lambda\!\left[\alpha\|\beta\|_1 + (1-\alpha)\|\beta\|_2^2\right]$$

Two hyperparameters:
- $\lambda$: overall regularisation strength
- $\alpha \in [0,1]$: mix between Lasso ($\alpha=1$) and Ridge ($\alpha=0$)

### 5.4.2 When to Use Elastic Net

**Lasso weakness**: with groups of correlated predictors, Lasso tends to arbitrarily pick one and drop the rest — even if all are genuinely relevant. This is unstable.

**Ridge weakness**: cannot produce sparse solutions (no exact zeros).

**Elastic Net** resolves this: the L2 term groups correlated predictors together (shrinks them similarly), while the L1 term encourages sparsity.

**Rule of thumb**:
- Dense signal, correlated predictors → Ridge
- Sparse signal, independent predictors → Lasso
- Sparse signal, correlated predictors → Elastic Net

### 5.4.3 Hyperparameter Selection

Use **cross-validation** over a 2D grid $(\lambda, \alpha)$.
sklearn's `ElasticNetCV` and `GridSearchCV` handle this efficiently.

In [ ]:
globals().update(run_cell(81, globals()))


# 6. Bayesian Linear Regression

## 6.1 The Bayesian Framework

### 6.1.1 Frequentist vs Bayesian

In frequentist statistics (OLS/MLE), $\beta$ is a **fixed unknown constant**.
We construct estimators and confidence intervals for this fixed quantity.

In Bayesian statistics, $\beta$ is a **random variable** with a **prior distribution** $p(\beta)$
reflecting our beliefs before seeing data. After observing data, we update to the **posterior**:

$$\underbrace{p(\beta \mid X, y)}_{\text{posterior}} \propto \underbrace{p(y \mid X, \beta)}_{\text{likelihood}} \cdot \underbrace{p(\beta)}_{\text{prior}}$$

This is Bayes' theorem. The posterior is our updated belief about $\beta$ after seeing the data.

### 6.1.2 Setup for Linear Regression

**Likelihood** (same as MLE):
$$p(y \mid X, \beta, \sigma^2) = \mathcal{N}(X\beta, \sigma^2 I)$$

**Prior** on $\beta$:
$$p(\beta) = \mathcal{N}(0, \tau^2 I) \quad \text{(zero-mean Gaussian, encodes "coefficients are small")}$$

**Posterior** (derived via Bayes):
$$p(\beta \mid X, y) = \mathcal{N}(\mu_n, \Sigma_n)$$

where:
$$\Sigma_n = \left(\frac{1}{\sigma^2}X'X + \frac{1}{\tau^2}I\right)^{-1},
\qquad \mu_n = \frac{1}{\sigma^2}\Sigma_n X'y$$

The posterior is again Gaussian — Gaussian priors are **conjugate** to the Gaussian likelihood.

### 6.1.3 Key Bayesian Outputs

| Quantity | Expression | Use |
|----------|-----------|-----|
| **Posterior mean** $\mu_n$ | $(\frac{1}{\sigma^2}X'X + \frac{1}{\tau^2}I)^{-1}\frac{1}{\sigma^2}X'y$ | Point estimate |
| **Posterior variance** $\Sigma_n$ | $(\frac{1}{\sigma^2}X'X + \frac{1}{\tau^2}I)^{-1}$ | Uncertainty in $\beta$ |
| **Credible interval** | Region containing 95% of posterior mass | "Probability $\beta$ is here" (valid claim!) |
| **Predictive distribution** | $p(\tilde{y} \mid \tilde{x}, X, y)$ | Uncertainty in new predictions |

In [ ]:
globals().update(run_cell(85, globals()))


## 6.2 MAP Estimation = Regularization

### 6.2.1 Maximum A Posteriori (MAP)

The **MAP estimate** maximises the posterior (instead of integrating it):

$$\hat{\beta}^{\text{MAP}} = \arg\max_\beta\; p(\beta \mid X, y)
= \arg\max_\beta \left[\log p(y \mid X, \beta) + \log p(\beta)\right]$$

### 6.2.2 Gaussian Prior → Ridge

With $\beta \sim \mathcal{N}(0, \tau^2 I)$:
$$\log p(\beta) = -\frac{1}{2\tau^2}\|\beta\|^2 + \text{const}$$

$$\hat{\beta}^{\text{MAP}} = \arg\min_\beta \underbrace{\|y - X\beta\|^2}_{\text{RSS}} + \underbrace{\frac{\sigma^2}{\tau^2}}_{\lambda}\|\beta\|^2$$

This is exactly **Ridge regression** with $\lambda = \sigma^2/\tau^2$.

$$\boxed{\text{Ridge} \equiv \text{MAP with Gaussian prior}}$$

### 6.2.3 Laplace Prior → Lasso

With $\beta_j \sim \text{Laplace}(0, b)$ (i.e., $p(\beta_j) \propto \exp(-|\beta_j|/b)$):
$$\log p(\beta) = -\frac{1}{b}\|\beta\|_1 + \text{const}$$

$$\hat{\beta}^{\text{MAP}} = \arg\min_\beta \|y - X\beta\|^2 + \frac{2\sigma^2}{b}\|\beta\|_1$$

This is exactly **Lasso** with $\lambda = 2\sigma^2/b$.

$$\boxed{\text{Lasso} \equiv \text{MAP with Laplace prior}}$$

### 6.2.4 The Unifying View

| Method | Prior on $\beta$ | Posterior mode |
|--------|-----------------|----------------|
| OLS | Flat (improper uniform) | MLE = OLS |
| Ridge | Gaussian $\mathcal{N}(0, \tau^2 I)$ | MAP = Ridge |
| Lasso | Laplace$(0, b)$ | MAP = Lasso |
| Full Bayesian | Any prior | Posterior distribution (not just mode) |

**Key difference**: MAP (Ridge/Lasso) gives a **point estimate**. Full Bayesian gives a **distribution**
— you get uncertainty quantification on the coefficients for free.

In [ ]:
globals().update(run_cell(88, globals()))


## 6.3 Full Bayesian Regression

### 6.3.1 Beyond the Posterior Mode

MAP gives us the **mode** of the posterior — a point estimate. Full Bayesian regression keeps the
**entire posterior distribution**, which gives us:

1. **Uncertainty in $\beta$**: the posterior covariance $\Sigma_n$ tells us how confident we should be
2. **Predictive distribution**: integrates over uncertainty in $\beta$ to give predictions with proper uncertainty

### 6.3.2 Conjugate Analysis (known $\sigma^2$)

With Gaussian prior and Gaussian likelihood, the posterior is Gaussian (conjugate pair):

$$p(\beta \mid X, y) = \mathcal{N}(\mu_n, \Sigma_n)$$

$$\Sigma_n = \left(\frac{X'X}{\sigma^2} + \frac{I}{\tau^2}\right)^{-1},
\qquad \mu_n = \frac{\Sigma_n X'y}{\sigma^2}$$

### 6.3.3 Predictive Distribution

For a new observation $\tilde{x}$, the **posterior predictive distribution** integrates over $\beta$:

$$p(\tilde{y} \mid \tilde{x}, X, y) = \int p(\tilde{y} \mid \tilde{x}, \beta)\; p(\beta \mid X, y)\; d\beta
= \mathcal{N}(\tilde{x}'\mu_n,\; \tilde{x}'\Sigma_n\tilde{x} + \sigma^2)$$

The predictive variance $\tilde{x}'\Sigma_n\tilde{x} + \sigma^2$ has two components:
- $\tilde{x}'\Sigma_n\tilde{x}$: **uncertainty in $\beta$** (shrinks as $n$ grows, goes to 0)
- $\sigma^2$: **irreducible noise** (remains even with infinite data)

### 6.3.4 Frequentist Confidence Intervals vs Bayesian Credible Intervals

**Frequentist 95% CI**: "If we repeated this experiment many times, 95% of the constructed intervals would contain the true $\beta$." Says nothing about *this* interval.

**Bayesian 95% Credible Interval**: "Given the data, there is 95% probability that $\beta$ lies in this region." The statement most people *think* frequentist CIs make.

The two intervals are numerically similar with uninformative priors (large $\tau^2$), but their interpretations differ fundamentally.

In [ ]:
globals().update(run_cell(91, globals()))


## Summary: The Linear Regression Landscape

| Topic | Key result | When it matters |
|-------|-----------|----------------|
| OLS | $\hat{\beta}=(X'X)^{-1}X'y$, minimises RSS | Default estimator, no distributional assumption |
| MLE | Identical $\hat{\beta}$ under $\varepsilon\sim\mathcal{N}$ | Likelihood-based model comparison (AIC/BIC) |
| Gauss-Markov | OLS is BLUE under A1–A5 | Justifies OLS efficiency, requires A4 most critically |
| Inference | $\hat{\beta}\sim\mathcal{N}$, use $t$/$F$ under A6 | Coefficient significance, model testing |
| Hat matrix | $H=X(X'X)^{-1}X'$, leverage $h_{ii}$ | Influence diagnostics, Cook's distance |
| OVB | Bias $=(X'X)^{-1}X'Z\gamma$ | Endogeneity detection and direction |
| Heteroscedasticity | OLS unbiased, SEs wrong → HC robust SEs | Cross-sectional data, variance grows with $x$ |
| Autocorrelation | OLS unbiased, SEs underestimated → Newey-West | Time-series regressions (finance!) |
| Ridge | $\hat{\beta}^R=(X'X+\lambda I)^{-1}X'y$ | Multicollinearity, dense signal |
| Lasso | $\ell_1$ penalty, sparsity | Many irrelevant predictors, feature selection |
| Elastic Net | $\ell_1 + \ell_2$, handles groups | Sparse signal + correlated predictors |
| MAP | Ridge $\equiv$ Gaussian prior; Lasso $\equiv$ Laplace prior | Unifies regularisation and Bayesian perspectives |
| Full Bayes | Posterior + predictive distributions | Uncertainty quantification in $\hat{\beta}$ |